In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes scikit-learn matplotlib seaborn
!pip install --upgrade transformers accelerate
import pandas as pd
import torch
from torch.utils.data import Dataset
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import ast
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
!pip install --upgrade transformers sentencepiece
from tqdm import tqdm
import re
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import login
token = ...         # PUT HERE YOUR ACCESS TOKEN FOR THE META-LLAMA/LLAMA-3.2-1B MODEL
login(token=token)

In [ ]:
MAX_LEN = 32
def preprocess(example):
    input_enc = tokenizer(example["input"], truncation=True, max_length=MAX_LEN)
    output_enc = tokenizer(example["output"], truncation=True, max_length=MAX_LEN)


    input_ids = input_enc["input_ids"] + output_enc["input_ids"]
    attention_mask = [1]*len(input_ids)

    # mask -100 input
    labels = [-100]*len(input_enc["input_ids"]) + output_enc["input_ids"] # vedi cosa fa

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.2-1B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none"
)
# cuda out of memory avoidance

model = get_peft_model(model, peft_config)
model.config.use_cache = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/PIZZA_CARBONARA/GENERATIVE_TASK/data/manzoni_train_tokens.csv")  # pd.read_csv("/content/drive/MyDrive/HW2_MNLP/manzoni_train_tokens.csv")
CHUNK_SIZE = 8

prompt_template = (
    "Split the text with [EoS] tokens:\n"
)
#Dataset definition
input_output_pairs = []

tokens = []
labels = []

for idx, row in df.iterrows():
    tokens.append(str(row["token"]))
    labels.append(int(row["label"]))

    if len(tokens) == CHUNK_SIZE:
        # INPUT: tokens
        input_text = prompt_template + " ".join(tokens) # define input as prompt + 8 read tokens

        # OUTPUT TARGET: with [EoS]
        output_text = ""
        for token, label in zip(tokens, labels):
            output_text += token + " "
            if label == 1:
                output_text += "[EoS] " # add [EoS] if label is 1


        input_output_pairs.append({
            "input": input_text.strip(),
            "output": output_text.strip()
        })


        tokens = []
        labels = []



In [ ]:
train_data, val_data = train_test_split(input_output_pairs, test_size=0.1, random_state=42)


train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

In [ ]:
tokenized_train = train_dataset.map(preprocess) # mapping using preprocess function
tokenized_val = val_dataset.map(preprocess)
# Collator for dynamic padding during training
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./llama-eos",
    eval_strategy="epoch",
    logging_dir="./logs",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=8,
    logging_strategy="steps",
    save_strategy="epoch",
    fp16=True,
    logging_steps=10,
    report_to="tensorboard",
    load_best_model_at_end=True,
    metric_for_best_model="loss", # cross entropy loss
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,

    data_collator = data_collator
)


trainer.train()

In [ ]:

pattern = r'\[EoS\]|[a-zA-Z0-9àèéìòùÈ\']+|[.,;:!?)(«»…–]'
model.eval()
df = pd.read_csv("/content/drive/MyDrive/PIZZA_CARBONARA_shared_folder/GENERATIVE_TASK/data/OOD_test.csv") # pd.read_csv("/content/drive/MyDrive/HW2_MNLP/manzoni_dev_tokens.csv")

tokens = []
labels = []
results = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing inputs"):
    tokens.append(str(row["token"]))
    labels.append(int(row["label"]))

    if len(tokens) == CHUNK_SIZE:
        # Build input
        input_text = prompt_template + " ".join(tokens) + " "

        # generate output
        encoded = tokenizer(input_text, return_tensors="pt").to("cuda")
        for i in range (5):
            input_text = prompt_template + " ".join(tokens) + " "

            with torch.no_grad():
                gen_ids = model.generate(
                    **encoded,
                    max_new_tokens=MAX_LEN,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
            generated_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
            generated_text = generated_text.strip()
            tokens_out = re.findall(pattern, generated_text)
            input_text = input_text.strip()
            input_text = re.findall(pattern, input_text)

            # Remove prompt from output
            if (tokens_out[0] =='Split'):
                tokens_out = tokens_out[len(input_text):]

            predicted_labels = []
            generated = []
            for i in range(len(tokens_out)):
                if tokens_out[i] == '[EoS]':
                    # if token is [EoS] it doesn't add anything to the labels list
                    continue
                if i < len(tokens_out)- 1 and tokens_out[i + 1] == '[EoS]':
                    predicted_labels.append(1)
                    generated.append(tokens_out[i])
                else:
                    predicted_labels.append(0)
                    generated.append(tokens_out[i])
            generated = generated[:CHUNK_SIZE]
            if(tokens == generated):
               break
            predicted_labels = predicted_labels[:CHUNK_SIZE]


        if len(predicted_labels) == len(labels):
            results.append({
                "input_tokens": tokens[:],
                "true_labels": labels[:],
                "predicted_labels": predicted_labels,
                "generated": generated[:]
            })
        else:
            results.append({
                "input_tokens": tokens[:],
                "true_labels": labels[:],
                "predicted_labels": np.zeros(CHUNK_SIZE),
                "generated": generated[:]
            })
        # Reset
        tokens = []
        labels = []



In [ ]:
df = pd.read_csv("/content/drive/MyDrive/PIZZA_CARBONARA_shared_folder/GENERATIVE_TASK/data/OOD_test.csv")  # check here
labels= []
tokens = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing inputs"):
    tokens.append(str(row["token"]))
    labels.append(int(row["label"]))

print(len(labels))

In [ ]:
#for the last 2 tokens

last_tokens = tokens[1520:]
last_labels = labels[1520:]

input_text = prompt_template + " ".join(last_tokens)
encoded = tokenizer(input_text, return_tensors="pt").to("cuda")

for i in range (5):
    input_text = prompt_template + " ".join(last_tokens) + " "

    with torch.no_grad():
        gen_ids = model.generate(
            **encoded,
            max_new_tokens=MAX_LEN,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
    print("generated:" , generated_text)
    generated_text = generated_text.strip()
    tokens_out = re.findall(pattern, generated_text)
    input_text = input_text.strip()
    input_text = re.findall(pattern, input_text)

    # Remove prompt from output
    if (tokens_out[0] =='Split'):
        tokens_out = tokens_out[len(input_text):]

    predicted_labels = []
    generated = []
    for i in range(len(tokens_out)):
        if tokens_out[i] == '[EoS]':
            # if token is [EoS] it doesn't add anything to the labels list
            continue
        if i < len(tokens_out)- 1 and tokens_out[i + 1] == '[EoS]':
            predicted_labels.append(1)
            generated.append(tokens_out[i])
        else:
            predicted_labels.append(0)
            generated.append(tokens_out[i])
    last_tokens_pred = generated[:2]
    if(tokens == generated):
        break
    last_labels_pred = predicted_labels[:2]


if len(last_labels_pred) == len(last_labels):
    results.append({
        "input_tokens": last_tokens[:],
        "true_labels": last_labels[:],
        "predicted_labels": last_labels_pred,
        "generated": last_tokens_pred[:]
    })
else:
    results.append({
        "input_tokens": last_tokens[:],
        "true_labels": last_labels[:],
        "predicted_labels": np.zeros(2),
        "generated": last_tokens_pred[:]
    })

In [ ]:
all_labels = []
all_preds = []

for r in results:

    for i in range (len(r["input_tokens"])):

      if(r["input_tokens"][i]==r["generated"][i]):
          all_labels.append(int(r["true_labels"][i]))
          all_preds.append(int(r["predicted_labels"][i]))
      else:
          all_labels.append(int(r["true_labels"][i]))
          all_preds.append(0)



# Accuracy standard
accuracy = accuracy_score(all_labels, all_preds)


precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

report = classification_report(all_labels, all_preds, digits=4, zero_division=0)


cm = confusion_matrix(all_labels, all_preds)


print(f"Accuracy: {accuracy:.4f}")
print(f"Precision (weighted): {precision:.4f}")
print(f"Recall (weighted): {recall:.4f}")
print(f"F1 Score (weighted): {f1:.4f}")
print("\nClassification Report:\n", report)
sns.heatmap(cm, fmt = 'd',annot=True, cmap="Blues", xticklabels=["Non-EOS", "EOS"], yticklabels=["Non-EOS", "EOS"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix Dev")
plt.show()

In [ ]:
tok = []
lab = []
for r in results:
    for i in range (len(r["input_tokens"])):
        if(r["input_tokens"][i]==r["generated"][i]):
            tok.append(r["input_tokens"][i])
            lab.append(int(r["predicted_labels"][i]))
        else:
            tok.append(r["input_tokens"][i])
            lab.append(0)



df = pd.DataFrame({'token': tok, 'label': lab})


df.to_csv('Pizza_carbonara-hw2_split-modelLlama.csv', index=False)